In [32]:
#=================================================
# Cellule "0" Explain Notebook LLM Visual Explorer (do NOT run)
#=================================================

Pipeline

Scenario      (Cell 3)  Load the default scenario from config.py
    ↓
Embeddings   (Cell 4)  Compute embedding vectors
    ↓
Projection   (Cell 5)  Project embeddings into a 3D PCA space
    ↓
Similarity   (Cell 6)  Compute cosine similarities
    ↓
DataFrame    (Cell 7)  Build the dataframe for visualization
    ↓
Visualization(Cell 8)  Display the interactive 3D scene

SyntaxError: invalid character '↓' (U+2193) (315839160.py, line 8)

In [1]:
# ==========================================================
# Cellule 1 - Initialization / environment
# ==========================================================

# Install required libraries
!pip -q install sentence-transformers plotly scikit-learn

In [2]:
#=================================================
# Cellule 2 - Initialization / imports
#=================================================

# Libraries only used by Notebook
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Clone the project (only if not already present)
import os

if not os.path.exists("/content/LLM-Visual-Explorer"):
    !git clone https://github.com/TintinDeBrest/LLM-Visual-Explorer.git

# Make the package visible to Python
import sys

PROJECT_DIR = "/content/LLM-Visual-Explorer"

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)


# Reload LlmExpl (only in dev phase - delete later)
import importlib

import explorer.embeddings
import explorer.projections
import explorer.plotting
import explorer.scenarios
import explorer.dataframe
import explorer.similarities
import explorer.display

importlib.reload(explorer.embeddings)
importlib.reload(explorer.projections)
importlib.reload(explorer.plotting)
importlib.reload(explorer.scenarios)
importlib.reload(explorer.dataframe)
importlib.reload(explorer.similarities)
importlib.reload(explorer.display)

# Import LlmExpl functions

from explorer.config import DEFAULT_SCENARIO
from explorer.config import MODEL_NAME
from explorer.scenarios import load_scenario
from explorer.display import display_scenario
from explorer.display import display_model
from explorer.embeddings import compute_embeddings
from explorer.projections import compute_pca
from explorer.plotting import plot_scene
from explorer.dataframe import create_dataframe
from explorer.similarities import compute_similarity
from explorer.similarities import rank_similarity_pairs


Cloning into 'LLM-Visual-Explorer'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 243 (delta 44), reused 3 (delta 3), pack-reused 168 (from 1)
Receiving objects: 100% (243/243), 80.34 KiB | 2.77 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [3]:
# ==========================================================
# Cellule 2 bis - Update from GitHub in case of modifs
# ==========================================================

%cd /content/LLM-Visual-Explorer

!git pull

/content/LLM-Visual-Explorer
Already up to date.


In [4]:
#=================================================
# Cellule 3 - Scenario
#=================================================

# Choix du scénario
SCENARIO_NAME = "animals" # Test du 01Aug26

scenario = load_scenario(SCENARIO_NAME)

display_scenario(scenario)

words = [
    obj["name"]
    for obj in scenario["objects"]
]

categories = [
    obj["category"]
    for obj in scenario["objects"]
]

SCÉNARIO : Animaux

Comparaison sémantique de cinq mammifères.

Objets analysés :

   • Chat
   • Lion
   • Chien
   • Loup
   • Renard

Nombre d'objets : 5


In [6]:
# ==========================================================
# Cellule 4 Chargement du modèle d'embeddings
# ==========================================================

from sentence_transformers import SentenceTransformer

# Chargement du modèle
model = SentenceTransformer(MODEL_NAME)

# Calcul des embeddings
embeddings = compute_embeddings(words)

# Affichage des informations
display_model(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODÈLE D'EMBEDDINGS

Architecture :
  SentenceTransformer

Modèle :
  sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Dimension des embeddings :
  384



In [7]:
#=================================================
# Cellule 5 - Projection
#=================================================

xyz, pca = compute_pca(embeddings)


In [8]:
#=================================================
# Cellule 6 - Dataframe
#=================================================

df = create_dataframe(
    words,
    categories,
    xyz,
)

In [9]:
#================================================
# Cellule 7 - Similarities
#================================================

similarities = compute_similarity(embeddings)


# From more to less similar
pairs = rank_similarity_pairs(
    words,
    similarities
)
print("Les PLUS similaires")
for word1, word2, score in pairs:
    print(f"{word1:10s} -    {word2:10s} : {score:.2f}")
print("Les MOINS similaires")

Les PLUS similaires
Chien      -    Loup       : 0.60
Chat       -    Loup       : 0.52
Loup       -    Renard     : 0.49
Lion       -    Loup       : 0.43
Chat       -    Chien      : 0.41
Chien      -    Renard     : 0.41
Lion       -    Renard     : 0.25
Chat       -    Renard     : 0.24
Lion       -    Chien      : 0.17
Chat       -    Lion       : 0.13
Les MOINS similaires


In [10]:
#==============================================
# Cellule 8 - Visualisation
#================================================



from explorer.plotting import CATEGORY_COLORS

print(CATEGORY_COLORS.keys())

#print(pio.renderers.default)

fig = plot_scene(df, SCENARIO_NAME)
fig.show()

print(type(fig.data[0]))

dict_keys(['Félin', 'Canidé'])


<class 'plotly.graph_objs._scatter3d.Scatter3d'>


In [11]:
# Test de santé du pipeline


print("=" * 60)
print("ÉTAT DU PIPELINE")
print("=" * 60)

print(f"Scénario          : {scenario['title']}")
print(f"Nombre d'objets   : {len(words)}")
print(f"Catégories        : {sorted(set(categories))}")
print(f"Modèle            : {MODEL_NAME}")
print(f"Embeddings        : {embeddings.shape}")
print(f"Projection        : {xyz.shape}")

print("=" * 60)

ÉTAT DU PIPELINE
Scénario          : Animaux
Nombre d'objets   : 5
Catégories        : ['Canidé', 'Félin']
Modèle            : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embeddings        : (5, 384)
Projection        : (5, 3)


In [12]:
# Avant le commit d'une version importante pour évrifier

!git diff --stat

In [ ]:
# Le commit lui-même pour figer cette version

!git add .
!git commit -m "LLM Visual Explorer V0.9.1 - stable release"
!git tag v0.9
!git push
!git push --tags